# 🧪 Model Sanity Check
Tests the full pipeline: data → features → model → predictions

## 1. Load Data & Verify Split

In [ ]:
import sys
sys.path.append('../src')  # adjust path if needed

from data import load_features, split_data
from models import train_baseline, make_X
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load the full feature dataset (from cache if available)
features_df = load_features()

print(f"Feature shape:   {features_df.shape}")
print(f"Columns:         {features_df.columns.tolist()}")
print(f"Goals:           {features_df['is_goal'].sum()} / {len(features_df)}")
print(f"Class balance:   {features_df['is_goal'].mean():.1%} positive (goals)")
features_df.head(3)

In [ ]:
# Split into train / validation / test sets
# Split is done by match (not by row) to avoid data leakage
train, val, test = split_data(features_df)

print(f"Train: {len(train):4d} shots | {train['is_goal'].mean():.1%} goals")
print(f"Val:   {len(val):4d} shots | {val['is_goal'].mean():.1%} goals")
print(f"Test:  {len(test):4d} shots | {test['is_goal'].mean():.1%} goals")

# Verify no match appears in more than one split (data leakage check)
train_ids = set(train['match_id'].unique())
val_ids   = set(val['match_id'].unique())
test_ids  = set(test['match_id'].unique())

assert len(train_ids & val_ids)   == 0, "❌ Overlap between Train and Val!"
assert len(train_ids & test_ids)  == 0, "❌ Overlap between Train and Test!"
assert len(val_ids   & test_ids)  == 0, "❌ Overlap between Val and Test!"
print("\n✅ No overlap between splits — split is clean")

## 2. Verify Feature Matrix

In [ ]:
# Build the feature matrix from the training set
X_train = make_X(train)
y_train = train['is_goal'].values

print(f"X_train shape:  {X_train.shape}")
print(f"y_train shape:  {y_train.shape}")

# Check for invalid values — these would break training
print(f"\nNaN in X:       {np.isnan(X_train).sum()}")
print(f"Inf in X:       {np.isinf(X_train).sum()}")

# First two features should be distance and angle
# Sanity check: are the ranges plausible for a football pitch?
print(f"\nDistance range: {X_train[:,0].min():.1f} – {X_train[:,0].max():.1f} m")
print(f"Angle range:    {X_train[:,1].min():.2f} – {X_train[:,1].max():.2f} rad")

In [ ]:
# Visualize a sample heatmap to confirm the freeze-frame encoding looks correct
# Expected: teammates/opponents scattered around the penalty area, shooter near goal
sample_heatmap = np.stack(train['heatmap'].values)[0]  # shape: (3, 68, 52)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, title, cmap) in enumerate(zip(
    axes,
    ['Channel 0: Teammates', 'Channel 1: Opponents', 'Channel 2: Shooter'],
    ['Blues', 'Reds', 'Greens']
)):
    ax.imshow(sample_heatmap[i], cmap=cmap, origin='lower', aspect='auto')
    ax.set_title(title)
    ax.axis('off')

plt.suptitle('Sample Heatmap (Gaussian blur applied)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Train Baseline Model

In [ ]:
# Train the baseline: Logistic Regression on distance + angle + flattened heatmap
# StandardScaler is applied inside the pipeline
print("Training baseline model...")
model = train_baseline(train)
print("✅ Training complete")
print(f"Model: {model.named_steps['clf']}")

## 4. Inspect Predictions

In [ ]:
# Run inference on the validation set
X_val = make_X(val)
y_val = val['is_goal'].values

# predict_proba returns [P(no goal), P(goal)] — we want the second column
y_proba = model.predict_proba(X_val)[:, 1]

# Print first 10 predictions side-by-side with ground truth
print("First 10 predictions vs. ground truth:\n")
for i in range(10):
    marker = '⚽' if y_val[i] == 1 else '  '
    print(f"  {marker}  xG: {y_proba[i]:.3f}  |  Goal: {bool(y_val[i])}")

In [ ]:
# Plot the xG distribution for goals vs. non-goals
# A good model should show goals clustering at higher xG values
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(y_proba[y_val == 0], bins=40, alpha=0.6, label='No Goal', color='steelblue')
ax.hist(y_proba[y_val == 1], bins=40, alpha=0.8, label='Goal',    color='red')
ax.set_xlabel('Predicted xG')
ax.set_ylabel('Number of shots')
ax.set_title('xG Distribution – Baseline Model (Validation Set)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Quick evaluation metrics — will be moved to evaluate.py later
# AUC:   measures ranking quality (higher = better, max 1.0)
# Brier: measures calibration / probability accuracy (lower = better, min 0.0)
from sklearn.metrics import roc_auc_score, brier_score_loss

auc   = roc_auc_score(y_val, y_proba)
brier = brier_score_loss(y_val, y_proba)

# Use StatsBomb's own xG as a strong reference baseline
sb_xg    = val['statsbomb_xg'].values
mask     = ~np.isnan(sb_xg)
auc_sb   = roc_auc_score(y_val[mask], sb_xg[mask])
brier_sb = brier_score_loss(y_val[mask], sb_xg[mask])

print(f"{'Model':<22} {'AUC':>6}  {'Brier':>7}")
print("-" * 38)
print(f"{'Baseline (ours)':<22} {auc:>6.3f}  {brier:>7.4f}")
print(f"{'StatsBomb xG':<22} {auc_sb:>6.3f}  {brier_sb:>7.4f}")
print()
